# <center>Modeling
This notebook will primarily contain the modeling algorithms used on the Student dataset. The dataset lends itself to regression and classification tasks, making it versatile for both avenues. 
## Outline:

1. Modeling objectives:
- Regression Task =   
    - **Can we predict a student's final grade?**  
- Classification Task =   
    - **Can we classify students according to their final academic performance?**  
    - **Can we predict whether a student will pass or fail?** 

2. Data preparation
    - Feature/target separation
    - Train/test split
    - Numerical features
    - Categorical features
    - Encoding
    - Preprocessing pipeline
    - Leakage considerations

3. Regression

    - 3.1 Regression objective
    - 3.2 Target: Final Grade (G3)
    - 3.3 Baseline model
    - 3.4 Linear Regression
    - 3.5 Decision Tree Regressor
    - 3.6 Random Forest Regressor
    - 3.7 Model evaluation
    - 3.8 Model comparison
    - 3.9 Regression findings

4. Classification

    - 4.1 Classification objective
    - 4.2 Creating Pass/Fail target
    - 4.3 Class distribution
    - 4.4 Baseline model
    - 4.5 Logistic Regression
    - 4.6 Decision Tree Classifier
    - 4.7 Random Forest Classifier
    - 4.8 Model evaluation
    - 4.9 Model comparison
    - 4.10 Classification findings

6. Overall Modeling Conclusions

7. Limitations

8. Recommendations / Future Work

In [1]:
# specify libraries to use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling libraries
# Baseline model
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Split the data
from sklearn.model_selection import train_test_split, GridSearchCV

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Random Forest model
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [2]:
# load the portuguese csv
port_df = pd.read_csv("student-por.csv", sep =';')
port_df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [3]:
# Open and read the txt file
with open('student.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Print the text to the VS Code terminal
print(content)


# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)
2 sex - student's sex (binary: "F" - female or "M" - male)
3 age - student's age (numeric: from 15 to 22)
4 address - student's home address type (binary: "U" - urban or "R" - rural)
5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)
6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)
7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or 

In [4]:
port_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object
 11  guardian    649 non-null    object
 12  traveltime  649 non-null    int64 
 13  studytime   649 non-null    int64 
 14  failures    649 non-null    int64 
 15  schoolsup   649 non-null    object
 16  famsup      649 non-null    object
 17  paid        649 non-null    object
 18  activities  649 non-null    object
 19  nursery     649 non-null    object
 20  higher    

In [5]:
port_df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

## Baseline Models
### 1. Dummy Regressor - Naive baseline
**What if we completely ignored student characteristics and just predicted the average grade for everyone?**    
- It essentially predicts the training-set mean for every student
- It doesn't look at age, study time, failures, absences, parental education, etc.
- Gives a reference point for judging whether the actual regression model has learned anything useful

In [6]:
# Define predictors and target variable
X = port_df.drop(columns=['G1', 'G2', 'G3']) 
# G1 and G2 were excluded from the predictors because they are previous-period grades and are highly predictive of the final grade (G3). 
# Excluding them allows the model to investigate whether student demographic, academic, family, and behavioral characteristics can predict final performance without relying directly on prior grades.
y = port_df['G3']

# Outline numerical and categorical variables
# Keep the ordinal variables as integers for the Dummy Regressor and first Linear Regression baseline
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
       'failures', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences']
categorical_features = ['school', 'sex', 'address', 'famsize', 'Pstatus', 
       'Mjob', 'Fjob', 'reason', 'guardian',  'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic']

# preprocessing
# To Note: 
# because the Dummy Regressor ignores the predictors, the preprocessing isn't actually necessary for this particular model. 
# But keeping the same pipeline structure is useful because I'll replace DummyRegressor with LinearRegression, DecisionTreeRegressor, etc. later. 
# It keeps the workflow consistent and prevents me from accidentally changing preprocessing between models.

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
# To Note:
# ordinary LinearRegression does not require standardization in the same way that Logistic Regression or regularized models do. 
# Regularization - prevent models from overfitting by adding a penalty for complexity (Ridge and Lasso)
# But keeping the scaler in the pipeline is useful because:
# - it puts numerical predictors on comparable scales;
# - it prevents preprocessing leakage;
# - it makes your pipeline consistent with later models such as Ridge/Lasso;
# - it allows you to change models without redesigning the preprocessing.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop = 'first',
        handle_unknown = 'ignore'
    ))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# model the data
dummy_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

# fit the model
dummy_model.fit(X_train, y_train)

# make predictions
dummy_y_pred = dummy_model.predict(X_test)

# evaluation
dummy_mae = mean_absolute_error(y_test, dummy_y_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_y_pred))
dummy_r2 = r2_score(y_test, dummy_y_pred)

print(f"Dummy MAE: {dummy_mae:.2f}")
print(f"Dummy RMSE: {dummy_rmse:.2f}")
print(f"Dummy R²: {dummy_r2:.2f}")


Dummy MAE: 2.39
Dummy RMSE: 3.17
Dummy R²: -0.03


- MAE  → how far predictions are from actual grades on average
- RMSE → whether larger errors are particularly substantial
- R²   → how much variation in grades is explained

### 2. Linear Regression - Baseline Predictive Model

In [7]:
# model the data
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()) # G3 is a numeric final grade (0–20)
])

# fit the model
baseline_model.fit(X_train, y_train)

# make predictions
baseline_y_pred = baseline_model.predict(X_test)

# evaluation
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_y_pred))
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.2f}")

Baseline MAE: 2.16
Baseline RMSE: 2.86
Baseline R²: 0.16


#### *Baseline Model Interpretation*

The Dummy Regressor provides a simple benchmark against which the Linear Regression model can be evaluated. The Dummy model achieved an MAE of **2.39**, RMSE of **3.17**, and R² of **-0.03**. Its negative R² indicates that predicting the average target value for every student performs slightly worse than using the mean as the reference level of variation, providing a relatively weak benchmark for the regression task.

The Linear Regression baseline achieved an MAE of **2.16**, RMSE of **2.86**, and R² of **0.16**. Compared with the Dummy Regressor, the baseline reduced MAE by **0.23 points** and RMSE by **0.31 points**, indicating that it makes somewhat smaller prediction errors.

The R² also improved from **-0.03 to 0.16**. This means that the Linear Regression model explains approximately **16% of the variation in students' final grades** on the test data, compared with the mean-based Dummy Regressor. However, a substantial proportion of the variation remains unexplained.

Overall, the baseline model performs better than the Dummy Regressor across all three evaluation metrics, confirming that the selected predictors contain some useful information for predicting final grades. However, the relatively low R² suggests that there is considerable room for improvement. Further analysis can therefore investigate feature engineering, appropriate treatment of categorical and ordinal variables, and alternative regression algorithms to determine whether predictive performance can be improved.

## Decision Tree Regressor
### Default Model

In [8]:
# model the data
tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor()) # G3 is a numeric final grade (0–20)
])

# fit the model
tree_model.fit(X_train, y_train)

# make predictions
tree_y_pred = tree_model.predict(X_test)

# evaluation
tree_mae = mean_absolute_error(y_test, tree_y_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_y_pred))
tree_r2 = r2_score(y_test, tree_y_pred)

print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")
print(f"Decision Tree R²: {tree_r2:.2f}")

Decision Tree MAE: 2.83
Decision Tree RMSE: 3.87
Decision Tree R²: -0.53


#### *Default Decision Tree Interpretation*

The Decision Tree Regressor achieved an MAE of **2.83**, RMSE of **3.87**, and R² of **-0.53** on the test set.

Compared with the Linear Regression baseline (MAE = 2.16, RMSE = 2.86, R² = 0.16), the Decision Tree produced larger prediction errors and explained less of the variation in students' final grades. Its performance was also only marginally better than the Dummy Regressor in terms of MAE, while its RMSE and R² were worse.

The negative R² of **-0.53** indicates that the untuned Decision Tree performed worse on the test data than the mean-based benchmark. This is possible when a model does not generalize well to unseen observations. The relatively higher RMSE also suggests that the tree made some larger prediction errors.

One possible explanation is that the default Decision Tree is too flexible and may be fitting patterns specific to the training data that do not generalize effectively to the test set. However, this result should be treated as a baseline finding rather than a definitive conclusion about Decision Trees. Hyperparameter tuning and further feature engineering can be explored later to determine whether a constrained tree can improve generalization.
